In [3]:
import os
import re
import string
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

# Download mandatory NLTK resources
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

# Ensure output repository directories exist
os.makedirs('results', exist_ok=True)

# ==========================================
# TASK 1: DATASET UNDERSTANDING
# ==========================================
print("--- TASK 1: DATASET UNDERSTANDING ---")

df = pd.read_csv('customer_support_text_classification.csv')

# 1. Number of records
num_records = len(df)
print(f"Total Number of Records: {num_records}")

# 2. Target labels/classes
target_classes = df['sentiment_label'].unique()
print(f"Target Classes: {target_classes}")

# 3. Sample text records
print("\nSample Text Records:")
print(df[['customer_message', 'sentiment_label']].head(3))

# 4. Average text length
df['raw_word_count'] = df['customer_message'].apply(lambda x: len(str(x).split()))
avg_text_length = df['raw_word_count'].mean()
print(f"\nAverage Text Length (Words): {avg_text_length:.2f}")

# 5. Class distribution
class_dist = df['sentiment_label'].value_counts()
class_dist_pct = df['sentiment_label'].value_counts(normalize=True) * 100
print("\nClass Distribution:")
for label in class_dist.index:
    print(f" - {label}: {class_dist[label]} ({class_dist_pct[label]:.2f}%)")


# ==========================================
# TASK 2: TEXT PREPROCESSING
# ==========================================
print("\n--- TASK 2: TEXT PREPROCESSING ---")

stop_words = set(stopwords.words('english'))

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'\d+', '', text)  # Remove numeric artifacts
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    cleaned_tokens = [word for word in tokens if word not in stop_words]
    return " ".join(cleaned_tokens)

df['cleaned_message'] = df['customer_message'].apply(clean_text)
print("Sample Preprocessing Output:")
print(f"Original: {df['customer_message'].iloc[0]}")
print(f"Cleaned:  {df['cleaned_message'].iloc[0]}")


# ==========================================
# TASK 3: TEXT VECTORIZATION & PREPARATION
# ==========================================
# Stratified split to preserve representation across sentiment dimensions
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['cleaned_message'], df['sentiment_label'], test_size=0.2, random_state=42, stratify=df['sentiment_label']
)

# Explicit Label Mapping
label_mapping = {'negative': 0, 'neutral': 1, 'positive': 2}
y_train_encoded = y_train.map(label_mapping).values
y_test_encoded = y_test.map(label_mapping).values

# Method A: TF-IDF Mapping (Baseline Matrix)
tfidf_vectorizer = TfidfVectorizer(max_features=2000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_raw)
X_test_tfidf = tfidf_vectorizer.transform(X_test_raw)

# Method B: Word Tokens Sequence Vectors (Deep Learning Array)
max_vocab = 3000
max_len = 20  # Selected near the upper-bound of average word lengths

tokenizer = Tokenizer(num_words=max_vocab, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_raw)

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train_raw), maxlen=max_len, padding='post', truncating='post')
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(X_test_raw), maxlen=max_len, padding='post', truncating='post')


# ==========================================
# TASK 4: BASELINE MODEL
# ==========================================
print("\n--- TASK 4: BASELINE MODEL EVALUATION ---")

baseline_model = LogisticRegression(max_iter=1000, random_state=42)
baseline_model.fit(X_train_tfidf, y_train)
baseline_preds = baseline_model.predict(X_test_tfidf)

print("Baseline Accuracy:", accuracy_score(y_test, baseline_preds))
print("\nClassification Report:\n", classification_report(y_test, baseline_preds))

# Save Confusion Matrix Visual Asset
plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_test, baseline_preds, labels=['negative', 'neutral', 'positive']), 
            annot=True, fmt='d', cmap='Blues', xticklabels=['negative', 'neutral', 'positive'], yticklabels=['negative', 'neutral', 'positive'])
plt.title('Baseline Model Confusion Matrix')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('results/model_evaluation.png')
plt.close()


# ==========================================
# TASK 5: SEQUENCE MODEL (cuDNN Optimized LSTM)
# ==========================================
print("\n--- TASK 5: SEQUENCE MODEL (LSTM) TRAINING ---")

embedding_dim = 64

# Note: recurrent_dropout is removed to ensure the model uses fast GPU execution kernels.
# Standard Dropout is applied via an independent explicit layer instead.
lstm_model = Sequential([
    Embedding(input_dim=max_vocab, output_dim=embedding_dim, input_length=max_len),
    LSTM(64),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

lstm_model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

lstm_model.summary()

# Train Model
history = lstm_model.fit(
    X_train_seq, y_train_encoded,
    epochs=5,
    batch_size=32,
    validation_data=(X_test_seq, y_test_encoded),
    verbose=1
)

# Save Evaluation Sample Records File
test_samples = df.sample(5, random_state=42)
sample_seqs = pad_sequences(tokenizer.texts_to_sequences(test_samples['cleaned_message']), maxlen=max_len)
predictions_prob = lstm_model.predict(sample_seqs)
predicted_classes = np.argmax(predictions_prob, axis=1)

inv_label_mapping = {0: 'negative', 1: 'neutral', 2: 'positive'}

with open('results/sample_predictions.txt', 'w') as f:
    f.write("Sample Predictions Output\n")
    f.write("=========================\n\n")
    for i, row in enumerate(test_samples.itertuples()):
        pred_label = inv_label_mapping[predicted_classes[i]]
        f.write(f"Message: {row.customer_message}\n")
        f.write(f"True Sentiment: {row.sentiment_label}\n")
        f.write(f"Predicted Sentiment: {pred_label}\n")
        f.write("-" * 40 + "\n")

print("\nAll pipeline tasks executed successfully. Core files written directly to 'results/'.")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\nehaf\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\nehaf\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\nehaf\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


--- TASK 1: DATASET UNDERSTANDING ---
Total Number of Records: 1500
Target Classes: <StringArray>
['neutral', 'positive', 'negative']
Length: 3, dtype: str

Sample Text Records:
                                    customer_message sentiment_label
0  I need information about the payment process. ...         neutral
1      I need information about the payment process.         neutral
2  The refund process was fast and convenient. I ...        positive

Average Text Length (Words): 12.72

Class Distribution:
 - neutral: 524 (34.93%)
 - negative: 497 (33.13%)
 - positive: 479 (31.93%)

--- TASK 2: TEXT PREPROCESSING ---
Sample Preprocessing Output:
Original: I need information about the payment process. My ticket number is 78732. Please respond as soon as possible.
Cleaned:  need information payment process ticket number please respond soon possible

--- TASK 4: BASELINE MODEL EVALUATION ---
Baseline Accuracy: 1.0

Classification Report:
               precision    recall  f1-score   suppo

c:\Users\nehaf\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 7s 32ms/step - accuracy: 0.4458 - loss: 1.0354 - val_accuracy: 0.6800 - val_loss: 0.6659
Epoch 2/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.9058 - loss: 0.2749 - val_accuracy: 0.9933 - val_loss: 0.0331
Epoch 3/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 1.0000 - loss: 0.0052 - val_accuracy: 1.0000 - val_loss: 0.0017
Epoch 4/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 1.0000 - loss: 0.0016 - val_accuracy: 1.0000 - val_loss: 9.0468e-04
Epoch 5/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 1.0000 - loss: 0.0011 - val_accuracy: 1.0000 - val_loss: 5.8114e-04
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 409ms/step

All pipeline tasks executed successfully. Core files written directly to 'results/'.
